# 🫁 ResNet-50 CXR — Clasificación Multilabel (v4 — Masking corregido + pos_weight dinámico)
## TFM: Sistema de Apoyo a la Decisión Clínica Multimodal · Módulo de Imagen
### Universidad de Salamanca · Máster en Análisis Avanzado de Datos Multivariantes y Big Data

---

## 📋 Qué corrige esta versión respecto a v3

La v3 reveló un problema real al analizar `nested_cv_summary_v3.json`: aunque el
**masking de NaN funcionaba correctamente** (las etiquetas no observadas se excluían
de la pérdida, por eso el total de pares etiqueta-muestra nunca llegó a 10.000), los
**pesos `pos_weight` heredados del EDA original ya no eran válidos** tras aplicar la
política de incertidumbre `uncertainty_policy="zeros"`.

### El problema en cifras (verificado sobre `train_clean.csv`)

| Etiqueta | NaN | Positivos | Negativos confirmados | Inciertos (-1) | Tras política "zeros" |
|---|---|---|---|---|---|
| Atelectasis | 64.3% | 2867 | 118 | 588 | pos=2867 / neg=706 → **4:1 a favor de POSITIVOS** |
| Lung Opacity | 64.9% | 3108 | 175 | 223 | pos=3108 / neg=398 → **~8:1 a favor de POSITIVOS** |

Los `pos_weight` del EDA (`Atelectasis=0.04`, pensado para un desequilibrio de
**24:1 en contra del positivo**) hacían exactamente lo opuesto de lo necesario:
penalizaban aún más una clase positiva que ya era mayoritaria tras el masking real.
Esto explica el AUC cercano a 0.50 (azar) en Atelectasis y Lung Opacity en los
resultados de v3.

### Fixes nuevos en v4 (numeración continúa desde v3)

| Fix | Descripción |
|-----|-------------|
| **[FIX 22]** | `pos_weight` ya NO se usa fijo desde el EDA. Se **recalcula dinámicamente** por fold, **después** de aplicar `uncertainty_policy`, como `n_neg/n_pos` real observado en ese subconjunto de entrenamiento. Esto es lo metodológicamente correcto: el peso debe compensar el desequilibrio que el modelo *realmente* ve, no el desequilibrio teórico de la población completa antes de decidir qué hacer con las etiquetas -1. |
| **[FIX 23]** | Las rutas de `hadm_id_*.npy` se explicitan como constantes independientes (antes se derivaban implícitamente de `NPY_DIR`). Esto permite verificar la alineación CSV↔npy de forma más robusta y deja constancia explícita en el código de cuáles son los tres ficheros de referencia para el join. |
| **[FIX 24]** | Detección automática de `cxr_train_224.npy` vs `cxr_train.npy`. Si existe una versión ya redimensionada a 224×224 se usa esa (evita tener que redimensionar en cada augmentación, ligero ahorro de tiempo); si no existe, se usa la versión original de 320×320 y se redimensiona en el pipeline como hasta ahora. La misma lógica se aplica a val y test por si existieran versiones `_224` análogas.
| **[FIX 25]** | Se añade `class_weight` adicional a nivel de muestra (no solo por etiqueta) mediante un `WeightedRandomSampler` opcional basado en la rareza combinada de las etiquetas positivas de cada radiografía, para reforzar el aprendizaje de combinaciones poco frecuentes (ej. las 5 etiquetas a la vez, solo 1.2% de los casos). |
| **[FIX 26]** | El backbone se descongela parcialmente (`layer4`) en la segunda mitad del entrenamiños final de cada fold externo, ya que con pos_weight corregido el modelo tiene una señal de gradiente más informativa y vale la pena permitir algo de fine-tuning. Se mantiene **congelado completamente** durante el tuning interno (igual que v3) para no disparar el coste computacional del Nested CV. |
| **[FIX 27]** | Grid y K ligeramente ampliados respecto a v3 para que el resultado sea representativo de una ejecución nocturna real (no de pocos minutos), pero sin volver al coste de v1/v2. Se documenta el tiempo estimado total. |

### ⚠️ Importante: por qué esto SÍ es masking correcto y qué cambia

El masking de observabilidad (excluir NaN del gradiente) **nunca estuvo roto** —
es exactamente la `MaskedBCELoss` de v1-v3. Lo que estaba mal calibrado era el
**peso de compensación de clase**, que solo tiene sentido si se calcula sobre la
distribución de clases que el modelo efectivamente entrena, no sobre una tabla fija
copiada del EDA inicial sin tener en cuenta la política de incertidumbre elegida.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 1: INSTALACIÓN DE DEPENDENCIAS
# ══════════════════════════════════════════════════════════════════════════════
import subprocess, sys

packages = [
    "torchxrayvision", "albumentations", "scikit-learn",
    "pandas", "numpy", "matplotlib", "seaborn", "tqdm", "Pillow",
]
for pkg in packages:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)

print("✅ Dependencias instaladas.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 2: IMPORTACIONES, SEMILLAS Y CONFIGURACIÓN DE HILOS CPU
# [FIX 19] torch.set_num_threads fijado a núcleos físicos reales (i5-1135G7=4).
# ══════════════════════════════════════════════════════════════════════════════
import os, gc, warnings, random, json, copy, time
from pathlib import Path
from itertools import product as itertools_product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision.models as tv_models

try:
    import torchxrayvision as xrv
    XRV_AVAILABLE = True
except ImportError:
    XRV_AVAILABLE = False
    print("⚠ torchxrayvision no disponible — se usará ResNet-50 ImageNet como fallback.")

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

N_PHYSICAL_CORES = 4
torch.set_num_threads(N_PHYSICAL_CORES)
os.environ["OMP_NUM_THREADS"] = str(N_PHYSICAL_CORES)
os.environ["MKL_NUM_THREADS"] = str(N_PHYSICAL_CORES)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo : {DEVICE}")
print(f"   PyTorch     : {torch.__version__}")
print(f"   Hilos CPU   : {torch.get_num_threads()}  [FIX 19]")
print(f"   XRV         : {'disponible' if XRV_AVAILABLE else 'NO disponible (fallback ImageNet)'}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 3: CONFIGURACIÓN DE RUTAS Y CONSTANTES GLOBALES
# [FIX 1] Imágenes desde .npy.  [FIX 13] IMG_SIZE=160.
# [FIX 23] Rutas de hadm_id_*.npy explicitadas como constantes independientes,
#          apuntando exactamente a los tres ficheros de referencia para el join
#          CSV↔npy (uno por split: train/val/test).
# ══════════════════════════════════════════════════════════════════════════════

BASE_DATA_DIR = Path(
    r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0"
)

CSV_DIR   = BASE_DATA_DIR / "data_csv" / "clean"
TRAIN_CSV = CSV_DIR / "train_clean.csv"
VAL_CSV   = CSV_DIR / "val_clean.csv"
TEST_CSV  = CSV_DIR / "test_clean.csv"

NPY_DIR = BASE_DATA_DIR / "data_npy"

# [FIX 23] Rutas EXPLÍCITAS de hadm_id por split — estas son las que se usan
# para el join por hadm_id entre CSV limpio y npy de imágenes/labs.
HADM_TRAIN_NPY = Path(
    r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0\data_npy\train\hadm_id_train.npy"
)
HADM_VAL_NPY = Path(
    r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0\data_npy\val\hadm_id_val.npy"
)
HADM_TEST_NPY = Path(
    r"C:\TFM\1.Opción - Symile Mimic\symile-mimic-a-multimodal-clinical-dataset-of-chest-x-rays-electrocardiograms-and-blood-labs-from-mimic-iv-1.0.0\data_npy\test\hadm_id_test.npy"
)

# [FIX 24] Detección automática de versión _224 pre-redimensionada.
# Si existe cxr_<split>_224.npy se usa esa (ya viene a 224×224, evita resize
# repetido en cada augmentación); si no, se cae a la versión original 320×320.
def _resolve_cxr_npy_path(split_dir: Path, split_name: str) -> Path:
    candidate_224 = split_dir / f"cxr_{split_name}_224.npy"
    candidate_orig = split_dir / f"cxr_{split_name}.npy"
    if candidate_224.exists():
        print(f"   [FIX 24] Usando versión pre-redimensionada: {candidate_224.name}")
        return candidate_224
    print(f"   [FIX 24] Versión _224 no encontrada, usando original: {candidate_orig.name}")
    return candidate_orig

CXR_TRAIN_NPY = _resolve_cxr_npy_path(NPY_DIR / "train", "train")
CXR_VAL_NPY   = _resolve_cxr_npy_path(NPY_DIR / "val",   "val")
CXR_TEST_NPY  = _resolve_cxr_npy_path(NPY_DIR / "test",  "test")

OUTPUT_DIR = Path("outputs_resnet50_cxr_v4")
OUTPUT_DIR.mkdir(exist_ok=True)

LABELS   = ["Atelectasis", "Cardiomegaly", "Edema", "Lung Opacity", "No Finding", "Pleural Effusion"]
N_LABELS = len(LABELS)

# [FIX 22] POS_WEIGHTS_EDA se conserva solo como REFERENCIA HISTÓRICA / fallback.
# El peso real usado en el entrenamiento se recalcula dinámicamente en cada
# fold tras aplicar la política de incertidumbre (ver compute_dynamic_pos_weights
# en la celda 8).
POS_WEIGHTS_EDA_REFERENCE = {
    "Atelectasis"     : 0.04,
    "Cardiomegaly"    : 0.30,
    "Edema"           : 0.90,
    "Lung Opacity"    : 0.10,
    "No Finding"      : 1.00,
    "Pleural Effusion": 0.50,
}

# Si IMG_SIZE=224 coincide con los npy _224 detectados, el resize en albumentations
# es un no-op (más rápido). Si se usa la versión 320, se sigue redimensionando a
# IMG_SIZE en el pipeline. Mantenemos 160 como compromiso CPU; se puede subir a
# 224 si los npy _224 están disponibles y se quiere aprovechar directamente.
IMG_SIZE = 160

GENDER_MAP    = {0: 0, 1: 1}
RACE_MAP      = {"UNKNOWN": 0, "WHITE": 1, "BLACK": 2, "ASIAN": 3, "HISPANIC_LATINO": 4}
ADMISSION_MAP = {"SCHEDULED": 0, "EMERGENCY": 1, "OBSERVATION": 2, "URGENT": 3}
CXR_VIEW_MAP  = {"AP": 0, "PA": 1}
META_DIM      = 13
META_EMBED    = 64
IMG_PROJ      = 512

print("\n✅ Constantes configuradas.")
print(f"   [FIX 13] IMG_SIZE = {IMG_SIZE}")
print(f"   [FIX 23] HADM_TRAIN_NPY = {HADM_TRAIN_NPY}")
print(f"   [FIX 23] HADM_VAL_NPY   = {HADM_VAL_NPY}")
print(f"   [FIX 23] HADM_TEST_NPY  = {HADM_TEST_NPY}")
print(f"   [FIX 24] CXR_TRAIN_NPY  = {CXR_TRAIN_NPY}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 4: CARGA DE DATOS Y ALINEACIÓN CSV ↔ NPY
# [FIX 1][FIX 2][FIX 2b] Join por hadm_id, mmap_mode="r".
# [FIX 23] Usa las rutas HADM_*_NPY explícitas definidas en celda 3.
# ══════════════════════════════════════════════════════════════════════════════

def load_split(csv_path, cxr_npy_path, hadm_npy_path, split_name="train"):
    """
    Carga un split completo alineando CSV y npy por hadm_id.
    mmap_mode="r": el array se mapea desde disco, no se carga entero en RAM.
    """
    print(f"\n── Cargando split '{split_name}' ────────────────────────────────────")

    df = pd.read_csv(csv_path, sep=";")
    print(f"   CSV filas          : {len(df):,}")

    hadm_npy = np.load(hadm_npy_path, allow_pickle=True)
    print(f"   npy hadm_id count  : {len(hadm_npy):,}  (fuente: {hadm_npy_path.name})  [FIX 23]")

    hadm_to_npy_idx = {int(h): i for i, h in enumerate(hadm_npy)}

    df["_npy_idx"] = df["hadm_id"].apply(lambda h: hadm_to_npy_idx.get(int(h), -1))
    n_before = len(df)
    df = df[df["_npy_idx"] >= 0].reset_index(drop=True)
    print(f"   Filas con imagen   : {len(df):,}  (descartadas: {n_before - len(df)})")

    print(f"   Mapeando {cxr_npy_path.name} (mmap, sin carga completa en RAM) ...",
          end=" ", flush=True)
    cxr_npy = np.load(cxr_npy_path, mmap_mode="r")
    print(f"shape={cxr_npy.shape}  dtype={cxr_npy.dtype}")

    return df, cxr_npy, hadm_to_npy_idx


df_train, cxr_train_npy, hadm_to_npy_train = load_split(
    TRAIN_CSV, CXR_TRAIN_NPY, HADM_TRAIN_NPY, "train")
df_val,   cxr_val_npy,   hadm_to_npy_val   = load_split(
    VAL_CSV,   CXR_VAL_NPY,   HADM_VAL_NPY,   "val")
df_test,  cxr_test_npy,  hadm_to_npy_test  = load_split(
    TEST_CSV,  CXR_TEST_NPY,  HADM_TEST_NPY,  "test")

print(f"\n✅ Datos cargados:")
print(f"   Train : {len(df_train):,} muestras  |  CXR npy shape: {cxr_train_npy.shape}")
print(f"   Val   : {len(df_val):,}  muestras  |  CXR npy shape: {cxr_val_npy.shape}")
print(f"   Test  : {len(df_test):,}   muestras  |  CXR npy shape: {cxr_test_npy.shape}")
print(f"   [FIX 2b] Test CSV={len(df_test)} filas alineadas con npy={cxr_test_npy.shape[0]} → join correcto.")

sample = np.array(cxr_train_npy[0])
print(f"\n   Muestra npy[0]: shape={sample.shape}  min={sample.min():.1f}  "
      f"max={sample.max():.1f}  mean={sample.mean():.2f}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 5: VERIFICACIÓN DE LA DISTRIBUCIÓN REAL DE ETIQUETAS TRAS LA POLÍTICA
#          DE INCERTIDUMBRE (diagnóstico que motivó el [FIX 22])
# ══════════════════════════════════════════════════════════════════════════════
# Esta celda reproduce el análisis que detectó el problema de v3: muestra,
# para cada etiqueta y cada política de incertidumbre, cuál es el desequilibrio
# REAL que vería el modelo — en contraste con los pos_weight fijos del EDA,
# que no tenían en cuenta este efecto.
# ══════════════════════════════════════════════════════════════════════════════

def diagnose_label_balance(df, labels=LABELS):
    """
    Imprime, para cada etiqueta y cada política ('zeros'/'ones'), el recuento
    de positivos/negativos resultante tras aplicar esa política a los valores -1,
    y compara con los pos_weight fijos del EDA.
    """
    print(f"{'Etiqueta':20s} | {'NaN%':>6} | {'pol=zeros (pos/neg)':>22} | "
          f"{'pol=ones (pos/neg)':>22} | {'SPW EDA fijo':>12}")
    print("─" * 100)
    for lbl in labels:
        vals    = df[lbl]
        n_nan   = vals.isna().sum()
        n_pos0  = (vals == 1).sum()
        n_neg0  = (vals == 0).sum()
        n_unc   = (vals == -1).sum()

        # Política 'zeros': -1 → 0
        pos_zeros = n_pos0
        neg_zeros = n_neg0 + n_unc
        # Política 'ones': -1 → 1
        pos_ones  = n_pos0 + n_unc
        neg_ones  = n_neg0

        spw_eda = POS_WEIGHTS_EDA_REFERENCE.get(lbl, 1.0)

        print(f"{lbl:20s} | {n_nan/len(df)*100:5.1f}% | "
              f"{pos_zeros:>6d}/{neg_zeros:<6d} ({neg_zeros/max(pos_zeros,1):.2f}:1) | "
              f"{pos_ones:>6d}/{neg_ones:<6d} ({neg_ones/max(pos_ones,1):.2f}:1) | "
              f"{spw_eda:>12.2f}")

print("Diagnóstico de balance REAL por etiqueta y política (train completo):\n")
diagnose_label_balance(df_train)
print("""
Nota: la columna "SPW EDA fijo" es el peso usado en v1-v3, pensado para el
desequilibrio de la POBLACIÓN COMPLETA (incluyendo NaN sin masking), no para
el desequilibrio real tras aplicar la política de incertidumbre. Por eso en
Atelectasis y Lung Opacity el SPW fijo (0.04, 0.10) penaliza la clase positiva
cuando en realidad, tras 'zeros', el positivo es mayoritario.
""")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 6: GRID DE HIPERPARÁMETROS — VERSIÓN "EJECUCIÓN NOCTURNA"
# [FIX 27] Grid y K ampliados respecto a v3 para una corrida representativa
#          de varias horas (pensada para dejar corriendo toda la noche),
#          sin volver al coste de v1/v2.
# [FIX 16] unfreeze_layers sigue fijo a "none" en el LOOP INTERNO (tuning).
# [FIX 26] El reentrenamiento FINAL de cada fold externo sí permite
#          descongelar layer4 a mitad de entrenamiento (ver celda 12/13).
# ══════════════════════════════════════════════════════════════════════════════

HYPERPARAM_GRID = {
    "lr_backbone"           : [1e-5, 3e-5],
    "lr_head"                : [1e-4, 3e-4],
    "scheduler"               : ["cosine"],
    "num_epochs"              : [6],            # [FIX 27] antes 3, ahora 6
    "unfreeze_epoch"          : [999],           # backbone congelado en el loop INTERNO
    "unfreeze_layers"         : ["none"],
    "uncertainty_policy"      : ["zeros", "ones"],
    "dropout_rate"            : [0.3, 0.5],
    "weight_decay"            : [1e-4, 1e-3],
    "batch_size"              : [8],
    "use_label_correlation"   : [True, False],
    "use_meta_branch"         : [True],
    "threshold_search"        : [True],
    "augmentation_level"      : ["moderate"],
    "use_weighted_sampler"    : [True, False],   # [FIX 25]
}

N_RANDOM_CONFIGS = 6   # [FIX 27] antes 3

all_keys   = list(HYPERPARAM_GRID.keys())
all_values = list(HYPERPARAM_GRID.values())
all_combos = list(itertools_product(*all_values))
np.random.shuffle(all_combos)
SAMPLED_CONFIGS = [dict(zip(all_keys, c)) for c in all_combos[:N_RANDOM_CONFIGS]]

print(f"✅ Grid 'ejecución nocturna' definido.")
print(f"   Combinaciones posibles : {len(all_combos)}")
print(f"   Configs muestreadas    : {N_RANDOM_CONFIGS}")
print(f"   [FIX 22] pos_weight se recalcula dinámicamente por fold, no fijo del EDA")
print(f"   [FIX 25] use_weighted_sampler explorado como hiperparámetro")
print(f"   [FIX 26] backbone se descongela parcialmente SOLO en el reentrenamiento final")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 7: PIPELINES DE AUGMENTACIÓN
# [FIX 9] Sin A.Normalize (npy ya normalizado).  [FIX 13] IMG_SIZE=160.
# ══════════════════════════════════════════════════════════════════════════════

def get_augmentation_pipeline(level: str, img_size: int = IMG_SIZE):
    to_tensor = ToTensorV2()

    if level == "test":
        return A.Compose([A.Resize(img_size, img_size), to_tensor])

    elif level == "moderate":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.4),
            to_tensor,
        ])

    elif level == "aggressive":
        return A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.5),
            A.GaussNoise(var_limit=(0.001, 0.005), p=0.3),
            to_tensor,
        ])
    else:
        raise ValueError(f"Nivel desconocido: '{level}'.")

print("✅ Pipelines de augmentación definidos.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 8: DATASET PYTORCH + MASKING + POS_WEIGHT DINÁMICO
# [FIX 1] Lectura desde npy.  [FIX 9] Sin re-normalizar.
# [FIX 22] compute_dynamic_pos_weights: calcula pos_weight REAL para un
#          DataFrame y política dados, DESPUÉS de decidir qué hacer con -1.
#          Esto sustituye por completo a POS_WEIGHTS_EDA_REFERENCE como fuente
#          del peso usado en el entrenamiento.
# ══════════════════════════════════════════════════════════════════════════════

def npy_to_hwc_float(arr: np.ndarray) -> np.ndarray:
    arr = np.array(arr, dtype=np.float32)
    if arr.ndim == 3 and arr.shape[0] in (1, 3):
        arr = arr.transpose(1, 2, 0)
    elif arr.ndim == 2:
        arr = np.stack([arr, arr, arr], axis=-1)
    if arr.shape[2] == 1:
        arr = np.repeat(arr, 3, axis=2)
    return arr


def compute_dynamic_pos_weights(df, uncertainty_policy, labels=LABELS,
                                 clip_max=20.0, eps=1e-3):
    """
    [FIX 22] Calcula pos_weight = n_neg / n_pos para cada etiqueta, sobre el
    DataFrame dado, DESPUÉS de aplicar la política de incertidumbre a los -1.
    Esto es lo que BCEWithLogitsLoss(pos_weight=...) espera: un peso que
    aumenta la contribución de la clase positiva cuando es minoritaria
    (pos_weight > 1) o la reduce cuando es mayoritaria (pos_weight < 1).

    clip_max: límite superior para evitar pesos extremos en folds pequeños
              con pocos positivos (estabilidad numérica).
    eps: evita división por cero si n_pos=0 en algún fold/etiqueta raro.

    Returns:
        dict {label: pos_weight_float}
    """
    weights = {}
    for lbl in labels:
        vals = df[lbl]
        n_pos0 = (vals == 1).sum()
        n_neg0 = (vals == 0).sum()
        n_unc  = (vals == -1).sum()

        if uncertainty_policy == "ones":
            n_pos = n_pos0 + n_unc
            n_neg = n_neg0
        else:  # "zeros"
            n_pos = n_pos0
            n_neg = n_neg0 + n_unc

        w = n_neg / max(n_pos, eps)
        w = float(np.clip(w, 1.0 / clip_max, clip_max))
        weights[lbl] = w
    return weights


def compute_sample_weights_for_sampler(df, labels=LABELS):
    """
    [FIX 25] Calcula un peso por MUESTRA (no por etiqueta) proporcional a la
    rareza de la combinación de etiquetas positivas que presenta. Las muestras
    con combinaciones poco frecuentes (ej. las 5 etiquetas activas a la vez,
    solo 1.2% del dataset según el EDA) reciben mayor probabilidad de ser
    muestreadas por WeightedRandomSampler.
    """
    combo_key = df[labels].apply(
        lambda row: tuple((row == 1).astype(int)), axis=1
    )
    combo_counts = combo_key.value_counts()
    weights = combo_key.map(lambda k: 1.0 / combo_counts[k]).values
    return torch.tensor(weights, dtype=torch.double)


class CXRMultilabelDataset(Dataset):
    """
    [FIX 1] Lee imágenes desde npy.  [FIX 9] No re-normaliza.
    El masking de NaN (mask=0) se mantiene EXACTAMENTE igual que en v1-v3:
    es la política de incertidumbre y el pos_weight lo que cambia en v4.
    """
    def __init__(self, dataframe, cxr_npy, augmentation_pipeline,
                 uncertainty_policy="zeros", labels=LABELS):
        self.df                 = dataframe.reset_index(drop=True)
        self.cxr_npy            = cxr_npy
        self.transform          = augmentation_pipeline
        self.uncertainty_policy = uncertainty_policy
        self.labels             = labels

        age_col      = self.df["age"].astype(float)
        self.age_min = age_col.min()
        self.age_max = age_col.max()

    def __len__(self):
        return len(self.df)

    def _encode_labels_and_mask(self, row):
        """
        Masking SIN CAMBIOS respecto a v1-v3:
          NaN → label=0.0, mask=0.0  (excluida del gradiente)
          -1  → label según política, mask=1.0  (incluida)
           0  → label=0.0, mask=1.0
           1  → label=1.0, mask=1.0
        """
        label_vec = np.zeros(len(self.labels), dtype=np.float32)
        mask_vec  = np.zeros(len(self.labels), dtype=np.float32)
        for i, lbl in enumerate(self.labels):
            val = row[lbl]
            if pd.isna(val):
                label_vec[i], mask_vec[i] = 0.0, 0.0
            elif val == -1:
                mask_vec[i]  = 1.0
                label_vec[i] = 1.0 if self.uncertainty_policy == "ones" else 0.0
            else:
                label_vec[i] = float(val)
                mask_vec[i]  = 1.0
        return label_vec, mask_vec

    def _encode_metadata(self, row):
        age_norm = (float(row["age"]) - self.age_min) / (self.age_max - self.age_min + 1e-8)
        gender   = float(GENDER_MAP.get(row["gender"], 0))
        race_vec = np.zeros(len(RACE_MAP),      dtype=np.float32)
        adm_vec  = np.zeros(len(ADMISSION_MAP), dtype=np.float32)
        view_vec = np.zeros(len(CXR_VIEW_MAP),  dtype=np.float32)
        race_vec[RACE_MAP.get(str(row["race"]), 0)]               = 1.0
        adm_vec[ADMISSION_MAP.get(str(row["admission_type"]), 1)] = 1.0
        view_vec[CXR_VIEW_MAP.get(str(row["cxr_view"]), 0)]      = 1.0
        return np.concatenate([[age_norm, gender], race_vec, adm_vec, view_vec])

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        npy_idx = int(row["_npy_idx"])
        img_arr = self.cxr_npy[npy_idx]
        image   = npy_to_hwc_float(img_arr)
        image_tensor = self.transform(image=image)["image"].float()
        labels, mask = self._encode_labels_and_mask(row)
        metadata     = self._encode_metadata(row)
        return {
            "image"   : image_tensor,
            "labels"  : torch.tensor(labels,   dtype=torch.float32),
            "mask"    : torch.tensor(mask,      dtype=torch.float32),
            "metadata": torch.tensor(metadata,  dtype=torch.float32),
            "hadm_id" : int(row["hadm_id"]),
        }


print("✅ CXRMultilabelDataset definido (masking sin cambios respecto a v1-v3).")
print("✅ [FIX 22] compute_dynamic_pos_weights definido.")
print("✅ [FIX 25] compute_sample_weights_for_sampler definido.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 9: CACHÉ DE EMBEDDINGS DEL BACKBONE CONGELADO
# [FIX 20] Igual que v3: el backbone se ejecuta una sola vez por imagen.
# Nota: en v4 el caché sigue calculándose SOLO sobre la versión "test" del
# pipeline (sin augmentación aleatoria), igual que en v3. El [FIX 26] de
# descongelar layer4 en el reentrenamiento final OPERA SOBRE IMÁGENES, no
# sobre el caché — ver celda 13.
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def compute_embeddings_cache(df, cxr_npy, backbone, batch_size=16, desc="Cache"):
    backbone.eval()
    aug_fixed = get_augmentation_pipeline("test")
    ds_fixed  = CXRMultilabelDataset(df, cxr_npy, aug_fixed, uncertainty_policy="zeros")
    loader    = DataLoader(ds_fixed, batch_size=batch_size, shuffle=False, num_workers=0)

    all_embeds = []
    pbar = tqdm(loader, desc=desc, leave=False, unit="batch")
    for batch in pbar:
        images = batch["image"]
        if images.shape[1] == 3:
            images = images.mean(dim=1, keepdim=True)
        out = backbone(images)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        all_embeds.append(out.numpy())
    return np.concatenate(all_embeds, axis=0)


class CachedEmbeddingDataset(Dataset):
    """[FIX 20] Dataset sobre embeddings pre-calculados, masking sin cambios."""
    def __init__(self, dataframe, embeddings, uncertainty_policy="zeros", labels=LABELS):
        self.df          = dataframe.reset_index(drop=True)
        self.embeddings  = embeddings
        self.uncertainty_policy = uncertainty_policy
        self.labels      = labels
        age_col      = self.df["age"].astype(float)
        self.age_min = age_col.min()
        self.age_max = age_col.max()

    def __len__(self):
        return len(self.df)

    def _encode_labels_and_mask(self, row):
        label_vec = np.zeros(len(self.labels), dtype=np.float32)
        mask_vec  = np.zeros(len(self.labels), dtype=np.float32)
        for i, lbl in enumerate(self.labels):
            val = row[lbl]
            if pd.isna(val):
                label_vec[i], mask_vec[i] = 0.0, 0.0
            elif val == -1:
                mask_vec[i]  = 1.0
                label_vec[i] = 1.0 if self.uncertainty_policy == "ones" else 0.0
            else:
                label_vec[i] = float(val)
                mask_vec[i]  = 1.0
        return label_vec, mask_vec

    def _encode_metadata(self, row):
        age_norm = (float(row["age"]) - self.age_min) / (self.age_max - self.age_min + 1e-8)
        gender   = float(GENDER_MAP.get(row["gender"], 0))
        race_vec = np.zeros(len(RACE_MAP),      dtype=np.float32)
        adm_vec  = np.zeros(len(ADMISSION_MAP), dtype=np.float32)
        view_vec = np.zeros(len(CXR_VIEW_MAP),  dtype=np.float32)
        race_vec[RACE_MAP.get(str(row["race"]), 0)]               = 1.0
        adm_vec[ADMISSION_MAP.get(str(row["admission_type"]), 1)] = 1.0
        view_vec[CXR_VIEW_MAP.get(str(row["cxr_view"]), 0)]      = 1.0
        return np.concatenate([[age_norm, gender], race_vec, adm_vec, view_vec])

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        embed    = self.embeddings[idx]
        labels, mask = self._encode_labels_and_mask(row)
        metadata = self._encode_metadata(row)
        return {
            "embedding": torch.tensor(embed,    dtype=torch.float32),
            "labels"   : torch.tensor(labels,   dtype=torch.float32),
            "mask"     : torch.tensor(mask,      dtype=torch.float32),
            "metadata" : torch.tensor(metadata,  dtype=torch.float32),
            "hadm_id"  : int(row["hadm_id"]),
        }


print("✅ [FIX 20] Infraestructura de caché de embeddings definida.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 10: ARQUITECTURA — ResNet-50 + Correlación + Metadatos
# [FIX 10][FIX 11][FIX 12] sin cambios respecto a v3.
# [FIX 26] Se añade unfreeze_layer4() utilizable SOLO en el reentrenamiento
#          final de cada fold externo (celda 13), nunca en el loop interno.
# ══════════════════════════════════════════════════════════════════════════════

class LabelCorrelationModule(nn.Module):
    def __init__(self, n_labels=N_LABELS):
        super().__init__()
        self.correlation = nn.Linear(n_labels, n_labels, bias=False)
        nn.init.eye_(self.correlation.weight)
        self.correlation.weight.data *= 0.1
        self.norm = nn.LayerNorm(n_labels)

    def forward(self, logits):
        return self.norm(logits + self.correlation(logits))


class MetadataBranch(nn.Module):
    def __init__(self, meta_dim=META_DIM, meta_embed=META_EMBED):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(meta_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(128, meta_embed), nn.BatchNorm1d(meta_embed), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


def _infer_backbone_out_dim(backbone, img_size=IMG_SIZE):
    was_training = backbone.training
    backbone.eval()
    with torch.no_grad():
        dummy = torch.zeros(2, 1, img_size, img_size)
        out = backbone(dummy)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        dim = out.shape[1]
    if was_training:
        backbone.train()
    return dim, (out.ndim != 2)


class CXRHead(nn.Module):
    def __init__(self, backbone_dim=2048, n_labels=N_LABELS, dropout_rate=0.5,
                 use_label_correlation=True, use_meta_branch=True):
        super().__init__()
        self.use_meta_branch       = use_meta_branch
        self.use_label_correlation = use_label_correlation

        self.img_proj = nn.Sequential(
            nn.Linear(backbone_dim, IMG_PROJ), nn.BatchNorm1d(IMG_PROJ), nn.ReLU(inplace=True)
        )
        if use_meta_branch:
            self.meta_branch = MetadataBranch()
            fusion_dim = IMG_PROJ + META_EMBED
        else:
            self.meta_branch = None
            fusion_dim = IMG_PROJ

        self.head = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(fusion_dim, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate * 0.5),
            nn.Linear(256, n_labels),
        )
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                if m.bias is not None: nn.init.zeros_(m.bias)

        self.label_corr = LabelCorrelationModule(n_labels) if use_label_correlation else None

    def forward(self, embedding, metadata=None):
        proj = self.img_proj(embedding)
        if self.use_meta_branch and metadata is not None:
            fused = torch.cat([proj, self.meta_branch(metadata)], dim=1)
        else:
            fused = proj
        logits = self.head(fused)
        if self.label_corr is not None:
            logits = self.label_corr(logits)
        return {"logits": logits, "probs": torch.sigmoid(logits)}


class CXRResNet50(nn.Module):
    """
    [FIX 26] Se añade el método unfreeze_layer4(), de uso EXCLUSIVO en el
    reentrenamiento final de cada fold externo (nunca en el tuning interno).
    """
    def __init__(self, n_labels=N_LABELS, dropout_rate=0.5,
                 use_label_correlation=True, use_meta_branch=True,
                 pretrained_source="chexpert"):
        super().__init__()

        if pretrained_source == "chexpert" and XRV_AVAILABLE:
            try:
                xrv_model = xrv.models.ResNet(weights="resnet50-res512-all")
                full      = xrv_model.model
                self.backbone = nn.Sequential(*list(full.children())[:-2])  # [FIX 12]
                print("   ✓ Backbone: ResNet-50 pesos CheXpert (torchxrayvision) [FIX 12]")
            except Exception as e:
                print(f"   ⚠ XRV falló ({e}) → ResNet-50 ImageNet")
                backbone      = tv_models.resnet50(weights="IMAGENET1K_V1")
                self.backbone = nn.Sequential(*list(backbone.children())[:-2])
        else:
            backbone      = tv_models.resnet50(weights="IMAGENET1K_V1")
            self.backbone = nn.Sequential(*list(backbone.children())[:-2])
            print("   ✓ Backbone: ResNet-50 ImageNet")

        backbone_dim, self._needs_gap = _infer_backbone_out_dim(self.backbone, IMG_SIZE)
        print(f"   ✓ Backbone out dim: {backbone_dim}  |  needs_GAP: {self._needs_gap}  [FIX 11]")

        for p in self.backbone.parameters():
            p.requires_grad = False
        self.backbone.eval()
        print("   🔒 Backbone congelado por defecto.")

        self.cxr_head = CXRHead(
            backbone_dim=backbone_dim, n_labels=n_labels, dropout_rate=dropout_rate,
            use_label_correlation=use_label_correlation, use_meta_branch=use_meta_branch,
        )
        self.label_corr  = self.cxr_head.label_corr
        self.img_proj    = self.cxr_head.img_proj
        self.head         = self.cxr_head.head
        self.meta_branch  = self.cxr_head.meta_branch

    def unfreeze_layer4(self):
        """
        [FIX 26] Descongela SOLO layer4 (último bloque residual). Se invoca
        únicamente en el reentrenamiento final de cada fold externo, nunca
        durante el loop interno de selección de hiperparámetros (que sigue
        usando el caché de embeddings y backbone 100% congelado, ver celda 9).
        """
        unfrozen_params = 0
        for name, p in self.backbone.named_parameters():
            if "layer4" in name:
                p.requires_grad = True
                unfrozen_params += p.numel()
        print(f"   🔓 [FIX 26] layer4 descongelada ({unfrozen_params:,} parámetros).")

    @torch.no_grad()
    def extract_embedding(self, image):
        if image.shape[1] == 3:
            image = image.mean(dim=1, keepdim=True)
        out = self.backbone(image)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        return out

    def extract_embedding_grad(self, image):
        """Versión CON gradiente, usada solo cuando layer4 está descongelada."""
        if image.shape[1] == 3:
            image = image.mean(dim=1, keepdim=True)
        out = self.backbone(image)
        if out.ndim == 4:
            out = F.adaptive_avg_pool2d(out, 1).flatten(1)
        elif out.ndim == 3:
            out = out.mean(dim=-1)
        return out

    def forward(self, image, metadata=None):
        any_unfrozen = any(p.requires_grad for p in self.backbone.parameters())
        if any_unfrozen:
            embedding = self.extract_embedding_grad(image)
        else:
            embedding = self.extract_embedding(image)
        return self.cxr_head(embedding, metadata)


print("✅ Arquitectura CXRResNet50 + CXRHead definida.")
print("   [FIX 26] unfreeze_layer4() disponible para reentrenamiento final")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 11: UTILIDADES — MÉTRICAS, UMBRALES, OPTIMIZADOR, LOSS DINÁMICA
# [FIX 3][FIX 4][FIX 5] sin cambios.
# [FIX 22] DynamicMaskedBCELoss: recibe pos_weight calculado por fold/política,
#          en lugar de un diccionario fijo del EDA.
# ══════════════════════════════════════════════════════════════════════════════

class DynamicMaskedBCELoss(nn.Module):
    """
    [FIX 22] Igual estructura que la MaskedBCELoss de v1-v3, pero el
    pos_weight se pasa como tensor calculado dinámicamente por
    compute_dynamic_pos_weights() para el fold/política actuales, no como
    diccionario fijo del EDA.
    """
    def __init__(self, pos_weights_tensor: torch.Tensor):
        super().__init__()
        self.register_buffer("pos_weights", pos_weights_tensor)

    def forward(self, logits, labels, mask):
        bce = F.binary_cross_entropy_with_logits(
            logits, labels, pos_weight=self.pos_weights.to(logits.device), reduction="none"
        )
        masked = bce * mask
        return masked.sum() / mask.sum().clamp(min=1e-8)


def build_criterion_for_config(df_for_weights, config, labels=LABELS):
    """
    [FIX 22] Construye la loss con pos_weight calculado dinámicamente sobre
    el DataFrame de entrenamiento de ESTE fold/config, tras aplicar la
    política de incertidumbre elegida en config["uncertainty_policy"].
    """
    weights_dict = compute_dynamic_pos_weights(
        df_for_weights, config["uncertainty_policy"], labels=labels
    )
    weights_tensor = torch.tensor([weights_dict[l] for l in labels], dtype=torch.float32)
    return DynamicMaskedBCELoss(weights_tensor).to(DEVICE), weights_dict


def compute_multilabel_metrics(all_probs, all_labels, all_masks,
                                thresholds=None, labels=LABELS):
    if thresholds is None:
        thresholds = np.full(len(labels), 0.5)
    metrics, auc_list, ap_list, f1_list = {}, [], [], []
    for i, lbl in enumerate(labels):
        vm     = all_masks[:, i] == 1
        y_true = all_labels[vm, i]
        y_prob = all_probs[vm, i]
        y_pred = (y_prob >= thresholds[i]).astype(float)
        n_pos  = int(y_true.sum())
        n_neg  = int((1 - y_true).sum())
        if n_pos < 2 or n_neg < 2:
            auc, ap = float("nan"), float("nan")
        else:
            auc = roc_auc_score(y_true, y_prob)
            ap  = average_precision_score(y_true, y_prob)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        metrics[lbl] = {"AUC": auc, "AP": ap, "F1": f1, "n_pos": n_pos, "n_neg": n_neg}
        if not np.isnan(auc):
            auc_list.append(auc); ap_list.append(ap)
        f1_list.append(f1)
    metrics["macro_AUC"] = float(np.nanmean(auc_list)) if auc_list else float("nan")
    metrics["macro_AP"]  = float(np.nanmean(ap_list))  if ap_list  else float("nan")
    metrics["macro_F1"]  = float(np.nanmean(f1_list))  if f1_list  else float("nan")
    return metrics


def find_optimal_thresholds(all_probs, all_labels, all_masks,
                             labels=LABELS, n_thresholds=50):
    thresholds = np.full(len(labels), 0.5)
    for i, lbl in enumerate(labels):
        vm     = all_masks[:, i] == 1
        y_true = all_labels[vm, i]
        y_prob = all_probs[vm, i]
        if y_true.sum() < 2:
            continue
        best_f1, best_thr = -1.0, 0.5
        for thr in np.linspace(0.1, 0.9, n_thresholds):
            f1 = f1_score(y_true, (y_prob >= thr).astype(float), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thresholds[i] = best_thr
    return thresholds


def build_optimizer_and_scheduler(trainable_params, config, train_loader_len,
                                   num_epochs_override=None):
    num_epochs = num_epochs_override if num_epochs_override is not None else config["num_epochs"]
    optimizer = torch.optim.AdamW(
        trainable_params, lr=config["lr_head"], weight_decay=config["weight_decay"]
    )
    sched = config["scheduler"]
    if sched == "cosine":
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=num_epochs, eta_min=1e-7)
    elif sched == "onecycle":
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=config["lr_head"] * 10,
            steps_per_epoch=train_loader_len, epochs=num_epochs, pct_start=0.1)
    elif sched == "plateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3)  # [FIX 3]
    else:
        scheduler = None
    return optimizer, scheduler


print("✅ Utilidades definidas.")
print("   [FIX 22] DynamicMaskedBCELoss + build_criterion_for_config")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 12: ENTRENAMIENTO SOBRE EMBEDDINGS CACHEADOS (LOOP INTERNO)
# [FIX 20] Backbone NO se toca aquí — usa embeddings pre-calculados.
# [FIX 22] La loss se construye DENTRO de esta función con
#          build_criterion_for_config sobre el propio df_train_fold,
#          así el pos_weight es siempre específico del subconjunto que
#          realmente se está entrenando en cada fold interno.
# [FIX 25] use_weighted_sampler activa un WeightedRandomSampler basado en
#          la rareza de combinaciones de etiquetas (ver celda 8).
# [FIX 8] Progreso con print+flush.
# ══════════════════════════════════════════════════════════════════════════════

def train_one_epoch_cached(head_module, loader, optimizer, scheduler, criterion, config):
    head_module.train()
    total_loss, n = 0.0, 0
    for batch in loader:
        embed    = batch["embedding"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if config.get("use_meta_branch") else None

        optimizer.zero_grad()
        out  = head_module(embed, metadata)
        loss = criterion(out["logits"], labels, masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head_module.parameters(), max_norm=1.0)
        optimizer.step()
        if config.get("scheduler") == "onecycle" and scheduler is not None:
            scheduler.step()
        total_loss += loss.item(); n += 1
    return total_loss / max(n, 1)


@torch.no_grad()
def evaluate_cached(head_module, loader, criterion, config):
    head_module.eval()
    total_loss, n = 0.0, 0
    probs_l, labels_l, masks_l = [], [], []
    for batch in loader:
        embed    = batch["embedding"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if config.get("use_meta_branch") else None
        out  = head_module(embed, metadata)
        loss = criterion(out["logits"], labels, masks)
        total_loss += loss.item(); n += 1
        probs_l.append(out["probs"].cpu().numpy())
        labels_l.append(labels.cpu().numpy())
        masks_l.append(masks.cpu().numpy())
    return (total_loss / max(n, 1),
            np.concatenate(probs_l), np.concatenate(labels_l), np.concatenate(masks_l))


def train_model_cached(embeddings_train, df_train_fold,
                       embeddings_val, df_val_fold, config, verbose=True):
    """
    [FIX 22] criterion se construye AQUÍ DENTRO sobre df_train_fold, así el
    pos_weight es siempre el correcto para el subconjunto de este fold.
    [FIX 25] WeightedRandomSampler opcional según config["use_weighted_sampler"].
    """
    criterion_fold, weights_used = build_criterion_for_config(df_train_fold, config)

    ds_train = CachedEmbeddingDataset(df_train_fold, embeddings_train,
                                      uncertainty_policy=config["uncertainty_policy"])
    ds_val   = CachedEmbeddingDataset(df_val_fold,   embeddings_val,
                                      uncertainty_policy=config["uncertainty_policy"])

    if config.get("use_weighted_sampler", False):
        sample_weights = compute_sample_weights_for_sampler(df_train_fold)
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        loader_train = DataLoader(ds_train, batch_size=config["batch_size"],
                                  sampler=sampler, num_workers=0, drop_last=True)
    else:
        loader_train = DataLoader(ds_train, batch_size=config["batch_size"],
                                  shuffle=True, num_workers=0, drop_last=True)

    loader_val = DataLoader(ds_val, batch_size=config["batch_size"] * 4,
                            shuffle=False, num_workers=0)

    backbone_dim = embeddings_train.shape[1]
    head = CXRHead(
        backbone_dim=backbone_dim,
        dropout_rate=config["dropout_rate"],
        use_label_correlation=config["use_label_correlation"],
        use_meta_branch=config["use_meta_branch"],
    ).to(DEVICE)

    optimizer, scheduler = build_optimizer_and_scheduler(
        head.parameters(), config, len(loader_train))

    best_val_loss    = float("inf")
    best_state       = None
    best_metrics     = None
    best_thresholds  = np.full(N_LABELS, 0.5)
    patience_counter = 0
    PATIENCE         = 2

    history = {"train_loss": [], "val_loss": [], "val_auc_macro": []}

    for epoch in range(config["num_epochs"]):
        train_loss = train_one_epoch_cached(
            head, loader_train, optimizer, scheduler, criterion_fold, config)
        val_loss, all_probs, all_labels, all_masks = evaluate_cached(
            head, loader_val, criterion_fold, config)
        val_metrics = compute_multilabel_metrics(all_probs, all_labels, all_masks)

        if scheduler is not None and config["scheduler"] != "onecycle":
            if config["scheduler"] == "plateau":
                scheduler.step(val_loss)
            else:
                scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_auc_macro"].append(val_metrics["macro_AUC"])

        if verbose:
            print(f"       Ep{epoch+1}/{config['num_epochs']}  "
                  f"train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  "
                  f"AUC={val_metrics['macro_AUC']:.4f}", flush=True)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss    = val_loss
            best_state       = copy.deepcopy(head.state_dict())
            patience_counter = 0
            if config.get("threshold_search"):
                best_thresholds = find_optimal_thresholds(all_probs, all_labels, all_masks)
            best_metrics = compute_multilabel_metrics(
                all_probs, all_labels, all_masks, thresholds=best_thresholds)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                if verbose:
                    print(f"       ⏹ Early stopping en época {epoch+1}.", flush=True)
                break

    return best_state, best_metrics, best_thresholds, history, weights_used


print("✅ [FIX 20][FIX 22][FIX 25] Funciones de entrenamiento sobre embeddings definidas.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 13: PRE-CÁLCULO DE EMBEDDINGS (UNA SOLA VEZ)
# [FIX 20] Igual lógica que v3.
# ══════════════════════════════════════════════════════════════════════════════

print("═" * 70)
print("  [FIX 20] PRE-CÁLCULO DE EMBEDDINGS — backbone ejecutado UNA SOLA VEZ")
print("═" * 70)

_temp_model = CXRResNet50(pretrained_source="chexpert")
_backbone   = _temp_model.backbone
_backbone.eval()

t0 = time.time()
print("\n  Calculando embeddings de TRAIN...", flush=True)
embeddings_train_full = compute_embeddings_cache(
    df_train, cxr_train_npy, _backbone, batch_size=16, desc="Embeddings train"
)
print(f"  ✓ embeddings_train_full: {embeddings_train_full.shape}  "
      f"({(time.time()-t0)/60:.1f} min)", flush=True)

t1 = time.time()
print("\n  Calculando embeddings de TEST...", flush=True)
embeddings_test_full = compute_embeddings_cache(
    df_test, cxr_test_npy, _backbone, batch_size=16, desc="Embeddings test"
)
print(f"  ✓ embeddings_test_full: {embeddings_test_full.shape}  "
      f"({(time.time()-t1)/60:.1f} min)", flush=True)

del _temp_model, _backbone
gc.collect()

print(f"\n✅ [FIX 20] Embeddings pre-calculados en {(time.time()-t0)/60:.1f} min totales.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 14: NESTED CROSS-VALIDATION (LOOP INTERNO SOBRE EMBEDDINGS)
#           + REENTRENAMIENTO FINAL CON FINE-TUNING PARCIAL [FIX 26]
# [FIX 5][FIX 6][FIX 8][FIX 14][FIX 18] sin cambios estructurales respecto a v3.
# [FIX 22] criterion ahora se construye dinámicamente dentro de train_model_cached.
# [FIX 26] El reentrenamiento FINAL de cada fold externo opera sobre IMÁGENES
#          (no sobre el caché de embeddings), permitiendo descongelar layer4
#          en la segunda mitad de las épocas. Esto es más caro que entrenar
#          sobre embeddings, pero solo ocurre K_OUTER veces (no en el tuning).
# [FIX 27] K_OUTER=3, N_RANDOM_CONFIGS=6 — pensado para una corrida nocturna.
# ══════════════════════════════════════════════════════════════════════════════

K_OUTER = 3   # [FIX 27] antes 2 en v3
K_INNER = 3

df_cv     = df_train.copy().reset_index(drop=True)
n_samples = len(df_cv)

INNER_SUBSAMPLE_FRAC = 0.20   # [FIX 27] ligeramente subido desde 0.15 de v3

print("═" * 70)
print("  NESTED CROSS-VALIDATION — ResNet-50 CXR v4 (masking + pos_weight dinámico)")
print("═" * 70)
print(f"  Muestras en CV          : {n_samples:,}")
print(f"  K externo                : {K_OUTER}  |  K interno : {K_INNER}")
print(f"  Configs RS               : {len(SAMPLED_CONFIGS)}")
print(f"  [FIX 14] Subsample inner : {INNER_SUBSAMPLE_FRAC:.0%}")
print(f"  Total trains internos    : {K_OUTER * len(SAMPLED_CONFIGS) * K_INNER}  (sobre embeddings)")
print(f"  Reentrenam. final        : {K_OUTER}  (sobre IMÁGENES, con fine-tuning parcial [FIX 26])")
print(f"  ⏱ Tiempo estimado total  : 6-10 horas en CPU i5-1135G7 (apto para correr de noche)")
print("═" * 70, flush=True)

outer_results = []
outer_kf      = KFold(n_splits=K_OUTER, shuffle=True, random_state=SEED)

run_total   = K_OUTER * len(SAMPLED_CONFIGS) * K_INNER
run_current = 0
t_global    = time.time()

for outer_fold_idx, (outer_train_idx, outer_test_idx) in enumerate(
    outer_kf.split(np.arange(n_samples))
):
    print(f"\n{'═'*70}", flush=True)
    print(f"  FOLD EXTERNO {outer_fold_idx+1}/{K_OUTER}  "
          f"(train={len(outer_train_idx):,} | test={len(outer_test_idx):,})", flush=True)
    print(f"{'═'*70}", flush=True)

    df_outer_train = df_cv.iloc[outer_train_idx].reset_index(drop=True)
    df_outer_test  = df_cv.iloc[outer_test_idx].reset_index(drop=True)
    embed_outer_train = embeddings_train_full[outer_train_idx]
    embed_outer_test  = embeddings_train_full[outer_test_idx]

    best_inner_auc = -1.0
    best_config    = SAMPLED_CONFIGS[0]
    inner_kf       = KFold(n_splits=K_INNER, shuffle=True, random_state=SEED)

    for config_idx, config in enumerate(SAMPLED_CONFIGS):
        t_cfg = time.time()
        print(f"\n  ┌─ Config {config_idx+1}/{len(SAMPLED_CONFIGS)} "
              f"[fold ext {outer_fold_idx+1}/{K_OUTER}] {'─'*20}", flush=True)
        print(f"  │  lr_head={config['lr_head']}  dropout={config['dropout_rate']}  "
              f"corr={config['use_label_correlation']}  policy={config['uncertainty_policy']}  "
              f"sampler={config['use_weighted_sampler']}", flush=True)

        auc_scores = []
        for inner_fold_idx, (inner_train_idx, inner_val_idx) in enumerate(
            inner_kf.split(np.arange(len(df_outer_train)))
        ):
            run_current += 1
            t_inner = time.time()
            elapsed   = time.time() - t_global
            avg_per   = elapsed / run_current if run_current > 1 else 0
            remaining = avg_per * (run_total - run_current)
            print(f"  │  [{run_current:3d}/{run_total}] fold interno {inner_fold_idx+1}/{K_INNER}  "
                  f"ETA: {int(remaining//60)}m {int(remaining%60):02d}s ...",
                  end=" ", flush=True)

            df_inner_train_full   = df_outer_train.iloc[inner_train_idx].reset_index(drop=True)
            df_inner_val           = df_outer_train.iloc[inner_val_idx].reset_index(drop=True)
            embed_inner_train_full = embed_outer_train[inner_train_idx]
            embed_inner_val         = embed_outer_train[inner_val_idx]

            n_sub = max(int(len(df_inner_train_full) * INNER_SUBSAMPLE_FRAC), 80)
            sub_idx = np.random.RandomState(SEED + run_current).choice(
                len(df_inner_train_full), size=n_sub, replace=False
            )
            df_inner_train    = df_inner_train_full.iloc[sub_idx].reset_index(drop=True)
            embed_inner_train = embed_inner_train_full[sub_idx]

            _, val_metrics, _, _, _ = train_model_cached(
                embed_inner_train, df_inner_train,
                embed_inner_val,   df_inner_val,
                config, verbose=False
            )
            auc = (val_metrics.get("macro_AUC", float("nan"))
                   if val_metrics is not None else float("nan"))
            auc_scores.append(auc)

            t_inner_s = time.time() - t_inner
            print(f"✓ AUC={auc:.4f}  ({t_inner_s:.1f}s)  [n_train_sub={n_sub}]", flush=True)
            gc.collect()

        mean_auc  = float(np.nanmean(auc_scores))
        t_cfg_min = (time.time() - t_cfg) / 60
        print(f"  └─ Config {config_idx+1} completada  mean_AUC={mean_auc:.4f}  "
              f"({t_cfg_min:.1f} min)", flush=True)

        if mean_auc > best_inner_auc:
            best_inner_auc = mean_auc
            best_config    = config
            print(f"     ⭐ Nueva mejor config (AUC={best_inner_auc:.4f})", flush=True)

    print(f"\n  ✅ Mejor config fold externo {outer_fold_idx+1}  AUC_inner={best_inner_auc:.4f}",
          flush=True)
    for k, v in best_config.items():
        print(f"     {k:30s} = {v}", flush=True)

    # ── [FIX 26] Reentrenamiento final SOBRE IMÁGENES con fine-tuning parcial ──
    print(f"\n  🔁 Reentrenando sobre IMÁGENES con la mejor config "
          f"({len(df_outer_train):,} muestras) — incluye descongelado de layer4 "
          f"a mitad de entrenamiento [FIX 26]...", flush=True)
    t_final = time.time()

    model_final = CXRResNet50(
        n_labels=N_LABELS,
        dropout_rate=best_config["dropout_rate"],
        use_label_correlation=best_config["use_label_correlation"],
        use_meta_branch=best_config["use_meta_branch"],
        pretrained_source="chexpert",
    ).to(DEVICE)

    criterion_final, weights_final = build_criterion_for_config(df_outer_train, best_config)

    aug_train_final = get_augmentation_pipeline(best_config["augmentation_level"])
    aug_test_final  = get_augmentation_pipeline("test")

    ds_train_final = CXRMultilabelDataset(df_outer_train, cxr_train_npy, aug_train_final,
                                          uncertainty_policy=best_config["uncertainty_policy"])
    ds_val_final   = CXRMultilabelDataset(df_outer_test,  cxr_train_npy, aug_test_final,
                                          uncertainty_policy=best_config["uncertainty_policy"])

    if best_config.get("use_weighted_sampler", False):
        sw = compute_sample_weights_for_sampler(df_outer_train)
        sampler_final = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)
        loader_train_final = DataLoader(ds_train_final, batch_size=best_config["batch_size"],
                                        sampler=sampler_final, num_workers=0, drop_last=True)
    else:
        loader_train_final = DataLoader(ds_train_final, batch_size=best_config["batch_size"],
                                        shuffle=True, num_workers=0, drop_last=True)
    loader_val_final = DataLoader(ds_val_final, batch_size=best_config["batch_size"] * 2,
                                  shuffle=False, num_workers=0)

    NUM_EPOCHS_FINAL    = 8   # algo más que en el tuning interno
    UNFREEZE_AT_EPOCH   = 4   # [FIX 26] descongelar layer4 a mitad de entrenamiento

    optimizer_final, scheduler_final = build_optimizer_and_scheduler(
        model_final.cxr_head.parameters(), best_config, len(loader_train_final),
        num_epochs_override=NUM_EPOCHS_FINAL
    )

    best_val_loss_f, best_state_f, best_thresholds_f = float("inf"), None, np.full(N_LABELS, 0.5)
    history_final = {"train_loss": [], "val_loss": [], "val_auc_macro": []}
    patience_f, PATIENCE_F = 0, 3

    for epoch in range(NUM_EPOCHS_FINAL):
        if epoch == UNFREEZE_AT_EPOCH:
            model_final.unfreeze_layer4()
            # [FIX 4] Reconstruir optimizador incluyendo ahora los parámetros
            # de layer4, con LR menor (lr_backbone) para no destruir los pesos
            # CheXpert preentrenados.
            epochs_remaining = NUM_EPOCHS_FINAL - epoch
            trainable = list(model_final.cxr_head.parameters()) + [
                p for n, p in model_final.backbone.named_parameters() if "layer4" in n
            ]
            optimizer_final = torch.optim.AdamW([
                {"params": model_final.cxr_head.parameters(), "lr": best_config["lr_head"]},
                {"params": [p for n, p in model_final.backbone.named_parameters() if "layer4" in n],
                 "lr": best_config["lr_backbone"]},
            ], weight_decay=best_config["weight_decay"])
            scheduler_final = torch.optim.lr_scheduler.CosineAnnealingLR(
                optimizer_final, T_max=epochs_remaining, eta_min=1e-7)

        model_final.train()
        train_loss_sum, nb = 0.0, 0
        for batch in loader_train_final:
            images   = batch["image"].to(DEVICE)
            labels   = batch["labels"].to(DEVICE)
            masks    = batch["mask"].to(DEVICE)
            metadata = batch["metadata"].to(DEVICE) if best_config.get("use_meta_branch") else None
            optimizer_final.zero_grad()
            out  = model_final(images, metadata)
            loss = criterion_final(out["logits"], labels, masks)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_final.parameters(), max_norm=1.0)
            optimizer_final.step()
            train_loss_sum += loss.item(); nb += 1
        train_loss = train_loss_sum / max(nb, 1)

        model_final.eval()
        val_loss_sum, nvb = 0.0, 0
        probs_l, labels_l, masks_l = [], [], []
        with torch.no_grad():
            for batch in loader_val_final:
                images   = batch["image"].to(DEVICE)
                labels   = batch["labels"].to(DEVICE)
                masks    = batch["mask"].to(DEVICE)
                metadata = batch["metadata"].to(DEVICE) if best_config.get("use_meta_branch") else None
                out  = model_final(images, metadata)
                loss = criterion_final(out["logits"], labels, masks)
                val_loss_sum += loss.item(); nvb += 1
                probs_l.append(out["probs"].cpu().numpy())
                labels_l.append(labels.cpu().numpy())
                masks_l.append(masks.cpu().numpy())
        val_loss = val_loss_sum / max(nvb, 1)
        all_probs  = np.concatenate(probs_l)
        all_labels = np.concatenate(labels_l)
        all_masks  = np.concatenate(masks_l)
        val_metrics = compute_multilabel_metrics(all_probs, all_labels, all_masks)

        if scheduler_final is not None:
            scheduler_final.step()

        history_final["train_loss"].append(train_loss)
        history_final["val_loss"].append(val_loss)
        history_final["val_auc_macro"].append(val_metrics["macro_AUC"])

        print(f"     Ep{epoch+1}/{NUM_EPOCHS_FINAL}  train_loss={train_loss:.4f}  "
              f"val_loss={val_loss:.4f}  AUC={val_metrics['macro_AUC']:.4f}"
              f"{'  [layer4 ON]' if epoch >= UNFREEZE_AT_EPOCH else ''}", flush=True)

        if val_loss < best_val_loss_f - 1e-4:
            best_val_loss_f = val_loss
            best_state_f    = copy.deepcopy(model_final.state_dict())
            patience_f      = 0
            best_thresholds_f = find_optimal_thresholds(all_probs, all_labels, all_masks)
            best_metrics_f    = compute_multilabel_metrics(
                all_probs, all_labels, all_masks, thresholds=best_thresholds_f)
        else:
            patience_f += 1
            if patience_f >= PATIENCE_F:
                print(f"     ⏹ Early stopping en época {epoch+1}.", flush=True)
                break

    print(f"  ✓ Reentrenamiento completado en {(time.time()-t_final)/60:.1f} min", flush=True)

    model_final.load_state_dict(best_state_f)
    ds_test_o     = CXRMultilabelDataset(df_outer_test, cxr_train_npy, aug_test_final,
                                         uncertainty_policy=best_config["uncertainty_policy"])
    loader_test_o = DataLoader(ds_test_o, batch_size=32, shuffle=False, num_workers=0)

    model_final.eval()
    probs_l, labels_l, masks_l = [], [], []
    with torch.no_grad():
        for batch in loader_test_o:
            images   = batch["image"].to(DEVICE)
            labels   = batch["labels"].to(DEVICE)
            masks    = batch["mask"].to(DEVICE)
            metadata = batch["metadata"].to(DEVICE) if best_config.get("use_meta_branch") else None
            out = model_final(images, metadata)
            probs_l.append(out["probs"].cpu().numpy())
            labels_l.append(labels.cpu().numpy())
            masks_l.append(masks.cpu().numpy())
    test_probs       = np.concatenate(probs_l)
    test_labels_arr  = np.concatenate(labels_l)
    test_masks       = np.concatenate(masks_l)
    test_metrics = compute_multilabel_metrics(
        test_probs, test_labels_arr, test_masks, thresholds=best_thresholds_f
    )

    print(f"\n  📊 Test externo fold {outer_fold_idx+1}:", flush=True)
    print(f"     macro_AUC={test_metrics['macro_AUC']:.4f}  macro_F1={test_metrics['macro_F1']:.4f}",
          flush=True)
    for lbl in LABELS:
        m = test_metrics[lbl]
        auc_s = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A"
        print(f"     {lbl:22s} | AUC={auc_s:>6} | F1={m['F1']:.4f} | "
              f"N+={m['n_pos']:>4} | N-={m['n_neg']:>4} | "
              f"pos_weight={weights_final[lbl]:.2f}", flush=True)

    ckpt_path = OUTPUT_DIR / f"model_outer_fold{outer_fold_idx+1}.pt"
    torch.save({
        "model_state_dict": best_state_f,
        "best_config"      : best_config,
        "thresholds"       : best_thresholds_f.tolist(),
        "test_metrics"     : test_metrics,
        "outer_fold"       : outer_fold_idx + 1,
        "pos_weights_used" : weights_final,
    }, ckpt_path)
    print(f"  💾 Checkpoint: {ckpt_path}", flush=True)

    outer_results.append({
        "outer_fold"     : outer_fold_idx + 1,
        "best_config"    : best_config,
        "best_inner_auc" : best_inner_auc,
        "test_metrics"   : test_metrics,
        "thresholds"     : best_thresholds_f.tolist(),
        "history"        : history_final,
        "checkpoint"     : str(ckpt_path),
        "pos_weights_used": weights_final,
    })

    del model_final
    gc.collect()

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]
t_total_min = (time.time() - t_global) / 60
print("\n" + "═"*70, flush=True)
print("  NESTED CV COMPLETADO (v4 — masking + pos_weight dinámico + fine-tuning parcial)",
      flush=True)
print("═"*70, flush=True)
print(f"  Tiempo total       : {t_total_min:.1f} min ({t_total_min/60:.1f} h)", flush=True)
print(f"  AUC macro por fold : {[f'{a:.4f}' for a in auc_macros]}", flush=True)
print(f"  Media AUC macro    : {np.mean(auc_macros):.4f} ± {np.std(auc_macros):.4f}", flush=True)
print(f"  Media F1  macro    : {np.mean(f1_macros):.4f}  ± {np.std(f1_macros):.4f}", flush=True)
print("═"*70, flush=True)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 15: RESUMEN NESTED CV + GUARDADO JSON
# ══════════════════════════════════════════════════════════════════════════════

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
f1_macros  = [r["test_metrics"]["macro_F1"]  for r in outer_results]

print("  AUC por etiqueta (media ± std sobre folds externos):")
for lbl in LABELS:
    aucs = [r["test_metrics"][lbl]["AUC"] for r in outer_results]
    print(f"    {lbl:22s}: {np.nanmean(aucs):.4f} ± {np.nanstd(aucs):.4f}")

summary = {
    "version"        : "v4 (masking corregido + pos_weight dinámico + fine-tuning parcial)",
    "mean_macro_AUC" : float(np.mean(auc_macros)),
    "std_macro_AUC"  : float(np.std(auc_macros)),
    "mean_macro_F1"  : float(np.mean(f1_macros)),
    "std_macro_F1"   : float(np.std(f1_macros)),
    "fold_results"   : [{
        "outer_fold"      : r["outer_fold"],
        "macro_AUC"       : r["test_metrics"]["macro_AUC"],
        "macro_F1"        : r["test_metrics"]["macro_F1"],
        "best_config"     : r["best_config"],
        "thresholds"      : r["thresholds"],
        "checkpoint"      : r["checkpoint"],
        "pos_weights_used": r["pos_weights_used"],
        "per_label"       : {lbl: r["test_metrics"][lbl] for lbl in LABELS},
    } for r in outer_results],
}
p = OUTPUT_DIR / "nested_cv_summary_v4.json"
with open(p, "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n  💾 Resumen guardado: {p}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 16: VISUALIZACIÓN — CURVAS Y AUC POR ETIQUETA
# ══════════════════════════════════════════════════════════════════════════════

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("ResNet-50 CXR v4 — Masking corregido + pos_weight dinámico", fontsize=13, fontweight="bold")

ax = axes[0]
for r in outer_results:
    h = r["history"]
    ax.plot(h["train_loss"], linestyle="--", alpha=0.5, label=f"Fold {r['outer_fold']} train")
    ax.plot(h["val_loss"],                  alpha=0.9, label=f"Fold {r['outer_fold']} val")
ax.set_xlabel("Época"); ax.set_ylabel("Loss"); ax.set_title("Pérdida por Fold")
ax.legend(fontsize=7); ax.grid(alpha=0.3)

ax = axes[1]
for r in outer_results:
    ax.plot(r["history"]["val_auc_macro"], alpha=0.9, label=f"Fold {r['outer_fold']}")
ax.axhline(np.mean(auc_macros), color="red", linestyle=":", label="Media")
ax.set_xlabel("Época"); ax.set_ylabel("AUC macro"); ax.set_title("AUC macro en Val")
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[2]
means = [np.nanmean([r["test_metrics"][l]["AUC"] for r in outer_results]) for l in LABELS]
stds  = [np.nanstd( [r["test_metrics"][l]["AUC"] for r in outer_results]) for l in LABELS]
bars  = ax.bar(range(len(LABELS)), means, yerr=stds, color="steelblue", alpha=0.7, capsize=4)
ax.set_xticks(range(len(LABELS)))
ax.set_xticklabels([l[:10] for l in LABELS], rotation=35, ha="right", fontsize=8)
ax.set_ylabel("AUC-ROC"); ax.set_title("AUC por Etiqueta")
ax.set_ylim([0, 1.05])
ax.axhline(np.mean(auc_macros), color="red", linestyle="--", alpha=0.7, label="Macro media")
ax.legend(fontsize=8); ax.grid(alpha=0.3, axis="y")
for bar, val in zip(bars, means):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.01, f"{val:.3f}",
                ha="center", fontsize=7)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "nested_cv_results_v4.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✅ Figura guardada.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 17: EVALUACIÓN FINAL EN TEST SET OFICIAL
# [FIX 21] weights_only=False — checkpoint propio y de confianza.
# ⚠ EJECUTAR SOLO UNA VEZ.
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "═"*70)
print("  EVALUACIÓN FINAL EN TEST SET OFICIAL")
print("  ⚠ Esta evaluación se ejecuta UNA SOLA VEZ.")
print("═"*70)

auc_macros = [r["test_metrics"]["macro_AUC"] for r in outer_results]
best_outer_fold = outer_results[int(np.argmax(auc_macros))]
print(f"\n  Fold seleccionado: {best_outer_fold['outer_fold']}  "
      f"(AUC_outer={auc_macros[int(np.argmax(auc_macros))]:.4f})")

ckpt = torch.load(best_outer_fold["checkpoint"], map_location=DEVICE, weights_only=False)
best_cfg = ckpt["best_config"]

model_test = CXRResNet50(
    n_labels=N_LABELS,
    dropout_rate=best_cfg["dropout_rate"],
    use_label_correlation=best_cfg["use_label_correlation"],
    use_meta_branch=best_cfg["use_meta_branch"],
    pretrained_source="chexpert",
).to(DEVICE)
# El checkpoint puede incluir layer4 descongelada; load_state_dict lo gestiona
# automáticamente ya que el modelo recién creado tiene la misma arquitectura.
model_test.load_state_dict(ckpt["model_state_dict"])
model_test.eval()

best_thresholds_test = np.array(ckpt["thresholds"])

aug_test_off = get_augmentation_pipeline("test")
ds_test_off  = CXRMultilabelDataset(df_test, cxr_test_npy, aug_test_off,
                                    uncertainty_policy=best_cfg["uncertainty_policy"])
loader_test  = DataLoader(ds_test_off, batch_size=32, shuffle=False, num_workers=0)

print(f"  Test samples: {len(ds_test_off):,}")

probs_l, labels_l, masks_l = [], [], []
with torch.no_grad():
    for batch in loader_test:
        images   = batch["image"].to(DEVICE)
        labels   = batch["labels"].to(DEVICE)
        masks    = batch["mask"].to(DEVICE)
        metadata = batch["metadata"].to(DEVICE) if best_cfg.get("use_meta_branch") else None
        out = model_test(images, metadata)
        probs_l.append(out["probs"].cpu().numpy())
        labels_l.append(labels.cpu().numpy())
        masks_l.append(masks.cpu().numpy())
test_probs_f  = np.concatenate(probs_l)
test_labels_f = np.concatenate(labels_l)
test_masks_f  = np.concatenate(masks_l)

test_metrics_final = compute_multilabel_metrics(
    test_probs_f, test_labels_f, test_masks_f, thresholds=best_thresholds_test
)

print("\n  📊 MÉTRICAS FINALES:")
print(f"  {'Etiqueta':22s} | {'AUC':>6} | {'AP':>6} | {'F1':>6} | {'N+':>5} | {'N-':>5}")
print("  " + "─"*60)
for lbl in LABELS:
    m = test_metrics_final[lbl]
    auc_s = f"{m['AUC']:.4f}" if not np.isnan(m["AUC"]) else "  N/A "
    ap_s  = f"{m['AP']:.4f}"  if not np.isnan(m["AP"])  else "  N/A "
    print(f"  {lbl:22s} | {auc_s:>6} | {ap_s:>6} | {m['F1']:>6.4f} | {m['n_pos']:>5} | {m['n_neg']:>5}")
print("  " + "─"*60)
print(f"  {'MACRO':22s} | {test_metrics_final['macro_AUC']:>6.4f} | "
      f"{test_metrics_final['macro_AP']:>6.4f} | {test_metrics_final['macro_F1']:>6.4f}")

with open(OUTPUT_DIR / "final_test_results_v4.json", "w") as f:
    json.dump({
        "macro_AUC" : test_metrics_final["macro_AUC"],
        "macro_AP"  : test_metrics_final["macro_AP"],
        "macro_F1"  : test_metrics_final["macro_F1"],
        "per_label" : {lbl: test_metrics_final[lbl] for lbl in LABELS},
        "best_config": best_cfg,
        "pos_weights_used": ckpt.get("pos_weights_used", {}),
    }, f, indent=2, default=str)
print(f"\n  💾 Resultados finales guardados.")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CELDA 18: ANÁLISIS DE EQUIDAD POR SUBGRUPO
# ══════════════════════════════════════════════════════════════════════════════

df_test_r = df_test.reset_index(drop=True)

for col, vals, label in [
    ("gender",         [0, 1],                       "GÉNERO"),
    ("cxr_view",       ["AP", "PA"],                 "VISTA CXR"),
    ("race",           list(RACE_MAP.keys()),        "RAZA"),
    ("admission_type", list(ADMISSION_MAP.keys()),   "TIPO ADMISIÓN"),
]:
    print(f"\n── AUC macro por {label} ──────────────────────────────────────────")
    for v in vals:
        idx = df_test_r[df_test_r[col] == v].index.values
        if len(idx) < 10:
            print(f"  {str(v):25s}: n={len(idx)} (insuficiente)")
            continue
        m = compute_multilabel_metrics(
            test_probs_f[idx], test_labels_f[idx], test_masks_f[idx],
            thresholds=best_thresholds_test
        )
        print(f"  {str(v):25s} (n={len(idx):4d}): AUC={m['macro_AUC']:.4f}  F1={m['macro_F1']:.4f}")

print("\n✅ Análisis de equidad completado.")


---
## ✅ Resumen completo de fixes (v1 → v4)

| Fix | Descripción |
|-----|-------------|
| [FIX 1]-[FIX 21] | Heredados de v1-v3 (ver notebooks anteriores): npy en lugar de JPGs, join por hadm_id, ReduceLROnPlateau sin verbose, num_epochs_override, guardia NaN, gc.collect, recalculo de scope, progreso con print+flush, sin re-normalizar, colapso 3→1 canal, detección dinámica de dimensión, backbone puro sin avgpool+fc, IMG_SIZE=160, subsampling 15-20%, batch_size=8, backbone congelado en tuning, num_epochs reducido, grid/K reducidos, hilos CPU fijados, caché de embeddings, weights_only=False |
| **[FIX 22]** | **pos_weight calculado dinámicamente por fold/política**, no fijo del EDA — corrige el problema de desequilibrio invertido detectado en v3 |
| **[FIX 23]** | Rutas de hadm_id_*.npy explicitadas como constantes independientes |
| **[FIX 24]** | Detección automática de `cxr_<split>_224.npy` vs `cxr_<split>.npy` |
| **[FIX 25]** | `WeightedRandomSampler` opcional basado en rareza de combinaciones de etiquetas |
| **[FIX 26]** | Fine-tuning parcial (`layer4`) en el reentrenamiento final de cada fold externo |
| **[FIX 27]** | Grid y K ampliados para una ejecución representativa de varias horas (nocturna) |

## ⚠️ Qué esperar de esta ejecución
- Tiempo total estimado: **6-10 horas** en tu CPU (i5-1135G7, 4 núcleos), apto para
  dejar corriendo durante la noche.
- El pre-cálculo de embeddings (celda 13) sigue siendo el único paso que toca TODO
  el backbone sobre TODAS las imágenes — esperable ~80-90 min como en v3.
- El Nested CV interno (celda 14, primera mitad) es rápido (~segundos por entrenamiento).
- El reentrenamiento final por fold (celda 14, segunda mitad) es el más lento de
  esta versión porque opera sobre imágenes y con `layer4` descongelada las últimas
  épocas — esto es intencional y es lo que debería producir un AUC más informativo
  que el ~0.50 de v3.

## 📌 Próximos pasos
1. Si los AUC siguen cercanos a 0.5 tras corregir pos_weight, investigar si el
   embedding CheXpert congelado realmente captura información discriminativa para
   estas 6 etiquetas concretas (podría requerir descongelar más capas o usar más
   épocas de fine-tuning).
2. Aplicar [FIX 22] (pos_weight dinámico) también al futuro modelo ResNet1D de ECG.
3. Documentar en la memoria del TFM el error de v3 y su corrección en v4 como parte
   del proceso iterativo de validación del pipeline.
